> ### ⚠️ Select the **`Python 3 (croprow)`** kernel first
> This notebook runs in the isolated croprow env (Python **3.11**, OpenCV). If the first cell throws `ModuleNotFoundError: No module named 'cv2'`, the wrong interpreter is selected.
>
> **VSCode:** click **Select Kernel** (top-right) → **Jupyter Kernel** → **`Python 3 (croprow)`**, or **Python Environments... → Enter interpreter path...** and paste `croprow\.venv\Scripts\python.exe`.
>
> Do **not** use the repo-root `.venv` — that is the potato backend (Python 3.13, no cv2 by design). `croprow_disease` shares the `croprow` env; it needs no venv of its own.

# 08 — Package a portable, ready-to-train dataset

Builds a **self-contained** 2-class YOLO folder a trainer can unzip and train on directly — no need to run 01, no machine-specific absolute paths in the image lists. Copies the images, writes mirrored `healthy`/`unhealthy` box labels and a `data.yaml`, and drops in `set_yaml_path.py` and `README_TRAIN.md`.

The split matches 01 exactly (read from `data/train.txt` & `data/val.txt` when present). ~1.1 GB of images are copied.

> **This packages the LettuceMOTS bootstrap**, where classes are colour-derived and `unhealthy` is empty. `README_TRAIN.md` says so inside the bundle, so nobody downstream mistakes it for disease ground truth. A *provided* dataset (01b) is already a portable YOLO folder and needs no packaging — hand over that folder as it came.

In [ ]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow_disease/utils.py) so the package imports
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow_disease" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow_disease import utils as U
from croprow_disease.health import HealthParams

CW = REPO_ROOT / "croprow_disease"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

import shutil

# === CONFIG: the ONLY place to set the dataset location ===================
LETTUCE_ROOT_DEFAULT = r"D:\croprow_dataset\LettuceMOTS"
LETTUCE_ROOT = os.environ.get("LETTUCE_ROOT", LETTUCE_ROOT_DEFAULT)
PARAMS = HealthParams()

# Bundle output goes OUTSIDE the repo (it is large data).
OUT_DIR = Path(os.environ.get(
    "CROPROW_HEALTH_BUNDLE",
    str(Path(LETTUCE_ROOT).parent / "croprow_health_yolo")))
MAKE_ZIP = False   # True to also produce OUT_DIR.zip (PNGs barely compress)
# ==========================================================================

root = U.resolve_lettuce_root(LETTUCE_ROOT)
print("LETTUCE_ROOT :", root)
print("bundle out   :", OUT_DIR)

In [ ]:
SET_PATH_PY = '''"""Repoint data.yaml \'path:\' to THIS folder. Run once after unzipping:

    python set_yaml_path.py

Ultralytics resolves a relative \'path:\' against its global datasets_dir, not
the yaml location, so an absolute \'path:\' matching where the bundle actually
lives is the reliable, zero-surprise setup.
"""
from pathlib import Path

here = Path(__file__).resolve().parent
yml = here / "data.yaml"
lines = yml.read_text().splitlines()
out = [f"path: {here.as_posix()}" if ln.startswith("path:") else ln for ln in lines]
yml.write_text("\\n".join(out) + "\\n")
print("data.yaml \'path:\' set to", here)
'''
print("helper script prepared:", len(SET_PATH_PY), "chars")

## 1. Use the SAME split as notebook 01

Derive train/val sequences from the committed list files so the bundle is identical to what 01 produced; fall back to the seeded split if they are absent.

In [ ]:
def seqs_from_listfile(p):
    return sorted({Path(l).parent.name
                   for l in Path(p).read_text().splitlines() if l.strip()})

tr_txt, va_txt = DATA_DIR / "train.txt", DATA_DIR / "val.txt"
if tr_txt.is_file() and va_txt.is_file():
    train_seqs, val_seqs = seqs_from_listfile(tr_txt), seqs_from_listfile(va_txt)
    print("split source: data/train.txt + val.txt (matches 01)")
else:
    train_seqs, val_seqs = U.split_sequences(U.labeled_sequences(root),
                                             val_frac=0.25, seed=42)
    print("split source: seeded split_sequences (01 lists not found)")
print("TRAIN seqs:", train_seqs)
print("VAL   seqs:", val_seqs)
assert not (set(train_seqs) & set(val_seqs)), "sequence leaked across split!"

## 2. Package (copy images + write mirrored 2-class labels + data.yaml)

Classes are re-derived here from the polygons + pixels, so the bundle does not depend on 01 having run. Reads every frame — this takes minutes.

In [ ]:
manifest = U.package_dataset(root, OUT_DIR, train_seqs, val_seqs, params=PARAMS,
                             copy_images=True)
for split, info in manifest["splits"].items():
    print(f"{split:5s}: {info['images']:5d} images | healthy {info['healthy']:6d} | "
          f"unhealthy {info['unhealthy']:6d} | seqs={info['sequences']}")
print("\nlabel source:", manifest["label_source"])
print("data.yaml ->", manifest["data_yaml"])

## 3. Drop in the portability script + a README that states provenance

In [ ]:
totals = {
    "healthy": sum(i["healthy"] for i in manifest["splits"].values()),
    "unhealthy": sum(i["unhealthy"] for i in manifest["splits"].values()),
}
verdict = U.check_class_balance(totals)

README_TRAIN = f"""# croprow_disease — ready-to-train YOLO dataset (2 classes)

Self-contained detection dataset. Classes, **in this exact index order**:

    0 = healthy      1 = unhealthy

Boxes derived from the LettuceMOTS segmentation polygons; split is **by sequence
folder** (no frame leakage between train and val). Real data only.

## Provenance of the class labels — read this

{manifest["label_source"]}. The boxes are human annotations; the healthy /
unhealthy class is a **colour heuristic** applied to the real pixels inside each
polygon (green canopy -> healthy, brown/off -> unhealthy), not a verified
disease label. Any metric measured against these labels is agreement with that
rule, not disease-detection accuracy.

Instances too blurred, dark or small to judge on colour default to `healthy`
and are not counted as disease.

## Class balance in this bundle

    healthy   : {totals["healthy"]}
    unhealthy : {totals["unhealthy"]}

{verdict["message"]}

## Train

1. (once, after unzipping) fix the dataset path:
   ```
   python set_yaml_path.py
   ```
2. install ultralytics + a torch build matching your GPU (see pytorch.org).
3. train YOLO11n:
   ```
   yolo detect train data=data.yaml model=yolo11n.pt imgsz=640 epochs=100
   ```
   or open the repo's `croprow_disease/notebooks/03_train.ipynb` and set
   `DATA_YAML` to this folder's `data.yaml`.

## Layout
```
images/train/<seq>/*.png     labels/train/<seq>/*.txt   # "<0|1> cx cy w h"
images/val/<seq>/*.png       labels/val/<seq>/*.txt
data.yaml                     # nc=2, names=[healthy, unhealthy]
```
Ultralytics finds each label by swapping `images` -> `labels` in the image path.
"""

(OUT_DIR / "set_yaml_path.py").write_text(SET_PATH_PY)
(OUT_DIR / "README_TRAIN.md").write_text(README_TRAIN, encoding="utf-8")
print("wrote set_yaml_path.py and README_TRAIN.md\n")
print((OUT_DIR / "data.yaml").read_text())
print(verdict["message"])

## 4. Sanity-check the bundle (label lookup resolves) + optional zip

In [ ]:
# YOLO derives the label path by swapping images->labels; confirm it resolves.
sample_img = next((OUT_DIR / "images" / "train").rglob("*.png"))
sample_lab = Path(str(sample_img).replace(f"{os.sep}images{os.sep}",
                                          f"{os.sep}labels{os.sep}")).with_suffix(".txt")
print("sample image:", sample_img)
print("sample label:", sample_lab, "| exists:", sample_lab.is_file())

total_imgs = sum(1 for _ in (OUT_DIR / "images").rglob("*.png"))
total_labs = sum(1 for _ in (OUT_DIR / "labels").rglob("*.txt"))
print(f"bundle totals: {total_imgs} images, {total_labs} labels")
assert total_imgs == total_labs, "image/label count mismatch!"

# Every class id in the bundle must be 0 or 1.
bad = set()
for lab in (OUT_DIR / "labels").rglob("*.txt"):
    for line in lab.read_text().splitlines():
        if line.strip():
            cid = int(line.split()[0])
            if cid not in (0, 1):
                bad.add(cid)
assert not bad, f"out-of-range class ids in bundle: {bad}"
print("class ids OK (0/1 only)")

if MAKE_ZIP:
    zpath = shutil.make_archive(str(OUT_DIR), "zip",
                                root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name)
    print("zipped ->", zpath)
else:
    print("MAKE_ZIP is False -> folder bundle ready at", OUT_DIR)